# 02 — Train

Fine-tune the typed-decision model. The config is split in two:

- **architecture** (`DecisionModelConfig`, `src/config.py`) — encoder, head depth,
  sequence budgets; overridden per run with the `arch` block.
- **training loop** (`TrainingConfig`, `src/pipelines/config.py`) — dataset, optimizer,
  RLCD loss weights, callbacks; loaded from YAML and overridable here.

For a quick local check use `configs/rlcd_smoke.yaml` (tiny encoder); for a real run
use `configs/train.yaml`.


In [ ]:
from pathlib import Path
import os, sys

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("repo root:", REPO_ROOT)


## 1. Build the config

Keyword arguments win over the YAML. Comment the ones you do not want to override.
The E2 ablation knobs are exactly `w_rl` / `w_ce` / `sigma_start` / `sigma_end` /
`anneal_sigma` (`w_rl: 0` = CE-only, `w_ce: 0` = RL-only).


In [ ]:
import json
from dataclasses import asdict
from src.pipelines.config import load_training_config

cfg = load_training_config(
    REPO_ROOT / "configs" / "train.yaml",
    run_name="nb_demo",
    # epochs=3,
    # batch_size=16,
    # lr=2e-5,
    # w_rl=1.0, w_ce=1.0,
    # sigma_start=0.4, sigma_end=0.1, anneal_sigma=True,
    # arch={"encoder_name": "convaiinnovations/laya", "head_layers": 2},
    # max_examples=200,  # quick slice before committing to a full run
)
print(json.dumps(asdict(cfg), indent=2, default=str))


## 2. Train


In [ ]:
from src.pipelines.train import train

history, final = train(cfg)


## 3. Learning curves


In [ ]:
import matplotlib.pyplot as plt

epochs = [h["epoch"] for h in history]
plt.figure(figsize=(7, 4))
plt.plot(epochs, [h["loss"] for h in history], marker="o", label="train loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.grid(True)
plt.legend()
plt.show()

calib_epochs = [h["epoch"] for h in history if "raw_ece" in h]
if calib_epochs:
    plt.figure(figsize=(7, 4))
    plt.plot(calib_epochs, [h["raw_ece"] * 100 for h in history if "raw_ece" in h],
             marker="o", color="tab:red", label="calib raw ECE")
    plt.xlabel("epoch")
    plt.ylabel("raw ECE (%)")
    plt.grid(True)
    plt.legend()
    plt.show()


Checkpoints land in `checkpoints/<run_name>/` (`best.pt`, `epoch_*.pt`,
`train_log.json`); results in `results/<run_name>/`. Next: evaluate the checkpoint
in `03_evaluate_and_infer.ipynb`.
